# IndexTTS-vLLM on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/keboqi/index-tts-vllm/blob/master/index_tts_vllm_colab.ipynb)

Run the IndexTTS 2.0 speech studio with voice cloning, speaker presets, emotion controls, and streaming audio.
Select **Runtime → Change runtime type → G4 GPU** before running cells 1–6 in order.
The notebook requests G4 in its metadata and verifies the assigned RTX PRO 6000 GPU; availability depends on Colab.

Adapted from the supplied MiniMax H3 notebook: isolated Python 3.12, rerunnable setup, optional Google Drive persistence, and live server logs.
This app uses **FastAPI**. Cell 6 prints a Cloudflare public URL after the model is ready; keep that cell running while using the studio.
Anyone with the public URL can access the app and its files. Stop the cell to close the server and tunnel.

Code, environments, checkpoints, and caches stay on the Colab runtime disk. Optional Drive storage persists outputs and speaker presets.
Local runtime files disappear when Colab deletes the runtime.

In [ ]:
# @title 1. Verify G4 GPU and system resources
import os
import shutil
import subprocess
import sys
from pathlib import Path

NOTEBOOK_DIRECTORY = ""  # @param {type:"string"}
REQUIRE_G4 = True  # @param {type:"boolean"}
NOTEBOOK_DIR = Path(NOTEBOOK_DIRECTORY or globals().get("NOTEBOOK_DIR", Path.cwd())).expanduser().resolve()
if not NOTEBOOK_DIR.is_dir():
    raise FileNotFoundError(NOTEBOOK_DIR)
if sys.platform != "linux":
    raise RuntimeError("This notebook requires a Linux Colab GPU runtime.")
try:
    gpu_rows = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        text=True,
    ).strip().splitlines()
except (OSError, subprocess.CalledProcessError) as exc:
    raise RuntimeError("Select G4 GPU in Colab runtime settings, reconnect, and rerun.") from exc
if len(gpu_rows) != 1:
    raise RuntimeError(f"Expected one GPU, found: {gpu_rows}")
if REQUIRE_G4 and "RTX PRO 6000" not in gpu_rows[0].upper():
    raise RuntimeError(f"Expected Colab G4 / RTX PRO 6000, found: {gpu_rows[0]}")
print(f"GPU: {gpu_rows[0]}")
print(f"Notebook directory: {NOTEBOOK_DIR}")
for line in Path("/proc/meminfo").read_text().splitlines():
    if line.startswith("MemAvailable:"):
        print(f"Available system RAM: {int(line.split()[1]) / 2**20:.1f} GiB")
print(f"Free disk: {shutil.disk_usage(NOTEBOOK_DIR).free / 2**30:.1f} GiB")


### 2. Clone the repository

Clone the repository's default branch, or choose a branch/tag with REPO_REF.
Reruns reuse a complete checkout. Enable UPDATE_REPOSITORY to fast-forward a clean checkout to the selected remote revision.
Incomplete child checkouts are preserved under a backup name. Existing local changes are never reset.

In [ ]:
# @title 2. Clone or reuse IndexTTS-vLLM
import signal
import uuid

REPO_URL = "https://github.com/keboqi/index-tts-vllm.git"  # @param {type:"string"}
REPO_REF = ""  # @param {type:"string"}
UPDATE_REPOSITORY = False  # @param {type:"boolean"}

def stop_process_group(proc):
    # Signal even if the parent exited: workers may still belong to its group.
    try:
        os.killpg(proc.pid, signal.SIGTERM)
    except ProcessLookupError:
        return
    try:
        proc.wait(timeout=30)
    except subprocess.TimeoutExpired:
        pass
    try:
        os.killpg(proc.pid, signal.SIGKILL)
    except ProcessLookupError:
        pass
    proc.wait(timeout=5)

def run_logged(command, *, cwd=None, env=None):
    """Stream subprocess logs into Colab and stop child processes on interruption."""
    proc = subprocess.Popen(
        [str(part) for part in command], cwd=cwd, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace", bufsize=1,
        start_new_session=True,
    )
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
        result = proc.wait()
        if result:
            raise subprocess.CalledProcessError(result, command)
    finally:
        stop_process_group(proc)
        proc.stdout.close()

REQUIRED_FILES = (
    "quickstart.sh", "fastapi_webui_v2.py", "fastapi_webui_v2_impl.py",
    "requirements-core.txt", "requirements-modal.txt", "requirements-clearvoice.txt", "constraints-main.txt",
    "indextts_web/config.py", "indextts/infer_vllm_v2.py", "index_new.html",
)

def is_complete_repository(directory):
    if not (directory / ".git").exists():
        return False
    result = subprocess.run(
        ["git", "-C", str(directory), "ls-tree", "-r", "--name-only", "-z", "HEAD"],
        capture_output=True, text=True,
    )
    files = [name for name in result.stdout.split("\0") if name]
    return (
        result.returncode == 0 and set(REQUIRED_FILES).issubset(files)
        and all((directory / name).is_file() for name in files)
    )

WORKSPACE_DIR = NOTEBOOK_DIR if is_complete_repository(NOTEBOOK_DIR) else NOTEBOOK_DIR / "index-tts-vllm"
if not is_complete_repository(WORKSPACE_DIR):
    if WORKSPACE_DIR.exists() or WORKSPACE_DIR.is_symlink():
        backup = WORKSPACE_DIR.with_name(f"index-tts-vllm-incomplete-{uuid.uuid4().hex[:8]}")
        WORKSPACE_DIR.rename(backup)
        print(f"Preserved incomplete checkout: {backup}")
    command = ["git", "clone", "--depth", "1"]
    if REPO_REF.strip():
        command += ["--branch", REPO_REF.strip()]
    run_logged(command + [REPO_URL, WORKSPACE_DIR])
elif UPDATE_REPOSITORY:
    changes = subprocess.check_output(
        ["git", "-C", str(WORKSPACE_DIR), "status", "--porcelain"], text=True,
    )
    if changes.strip():
        raise RuntimeError("Checkout has local changes. Keep UPDATE_REPOSITORY off or handle them before updating.")
    run_logged(["git", "fetch", "--depth", "1", REPO_URL, REPO_REF.strip() or "HEAD"], cwd=WORKSPACE_DIR)
    run_logged(["git", "merge", "--ff-only", "FETCH_HEAD"], cwd=WORKSPACE_DIR)
if not is_complete_repository(WORKSPACE_DIR):
    raise RuntimeError("Checkout is incomplete. Inspect the clone logs before continuing.")
print(f"Workspace: {WORKSPACE_DIR}")
print("Revision:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=WORKSPACE_DIR, text=True).strip())


### 3. Install the isolated GPU environment

Use Python 3.12 and the repository's constraints: vLLM 0.10.2, Torch/torchaudio 2.8.0, torchvision 0.23.0, and CUDA 12.8.
This keeps vLLM's binary dependencies together instead of installing the latest Torch first.
[vLLM's installation guide](https://docs.vllm.ai/en/v0.10.2/getting_started/installation/gpu.html) requires CUDA 12.8 or newer for Blackwell.

Optional main-process integrations install from the same manifest used by the repo's Modal deployment.
Stable Audio source is pinned and uses that shared runtime stack. Its six older metadata pins are reported as known overrides; all other dependency-check errors stop setup.
ClearVoice installs in its own Python 3.12/CUDA environment by default, enabling reference enhancement and super-resolution. Its weights download on first use.
If INSTALL_CLEARVOICE is off, disable both enhancement options in the UI when adding a speaker.
MOSS and Qwen3-ASR + OmniVAD are installed in separate environments by default. MOSS uses the local Transformers service without Docker. Their model weights download on first use (no startup warmup).
If either ASR installation is disabled, select another transcription backend in the UI. Confucius and IndexTTS 2.5 require their own sibling repositories and setup.
First installation can take several minutes. Reruns reuse the environment; no Colab kernel restart is needed.

In [ ]:
# @title 3. Install Python 3.12 and compatible dependencies
INSTALL_OPTIONAL_FEATURES = True  # @param {type:"boolean"}
# @markdown Install ClearVoice for reference enhancement and super-resolution (separate environment).
INSTALL_CLEARVOICE = True  # @param {type:"boolean"}
# @markdown Install Docker-free MOSS and Qwen3-ASR + OmniVAD in separate environments. Weights download on first use.
INSTALL_MOSS = True  # @param {type:"boolean"}
INSTALL_QWEN_ASR = True  # @param {type:"boolean"}
INSTALL_FLASH_ATTENTION = True  # @param {type:"boolean"}

VENV_DIR = NOTEBOOK_DIR / "venv_index_tts"
CACHE_DIR = NOTEBOOK_DIR / ".cache_index_tts"
PYTHON = VENV_DIR / "bin/python"
RUN_ENV = os.environ.copy()
# Avoid importing Colab's preinstalled packages or notebook-only plotting backend.
for key in ("PYTHONPATH", "PYTHONHOME", "VIRTUAL_ENV", "MPLBACKEND", "PIP_CONSTRAINT", "UV_CONSTRAINT"):
    RUN_ENV.pop(key, None)
RUN_ENV.update({
    "PYTHONUNBUFFERED": "1", "PYTHONNOUSERSITE": "1",
    "UV_NO_CACHE": "1", "PIP_NO_CACHE_DIR": "1",
    "HF_HOME": str(CACHE_DIR / "huggingface"),
    "TORCH_HOME": str(CACHE_DIR / "torch"),
    "TRITON_CACHE_DIR": str(CACHE_DIR / "triton"),
    "VLLM_CACHE_ROOT": str(CACHE_DIR / "vllm"),
})
if os.geteuid() != 0:
    raise RuntimeError("System dependency setup expects the standard root Colab runtime.")
run_logged(["apt-get", "update"], env=RUN_ENV)
run_logged(["apt-get", "install", "-y", "--no-install-recommends",
            "git", "curl", "ffmpeg", "sox", "libsndfile1", "build-essential", "pkg-config"], env=RUN_ENV)
run_logged([sys.executable, "-m", "pip", "install", "--quiet", "uv"], env=RUN_ENV)
UV = [sys.executable, "-m", "uv"]
if not PYTHON.is_file():
    run_logged(UV + ["venv", "--python", "3.12", "--seed", str(VENV_DIR)], env=RUN_ENV)
version = subprocess.check_output([str(PYTHON), "-c", "import sys; print('%s.%s' % sys.version_info[:2])"], env=RUN_ENV, text=True).strip()
if version != "3.12":
    raise RuntimeError(f"{VENV_DIR} uses Python {version}; choose a fresh VENV_DIR for Python 3.12.")
RUN_ENV["VIRTUAL_ENV"] = str(VENV_DIR)
RUN_ENV["PATH"] = str(VENV_DIR / "bin") + os.pathsep + RUN_ENV["PATH"]

# Select CUDA for Torch packages only; general dependencies still come from PyPI.
# A global PyTorch --index-url hides pinned dependencies such as filelock.
constraints = WORKSPACE_DIR / "constraints-main.txt"
pip_install = UV + ["pip", "install", "--python", str(PYTHON),
                    "-c", str(constraints), "--torch-backend", "cu128"]
run_logged(pip_install + ["torch==2.8.0", "torchaudio==2.8.0", "torchvision==0.23.0"], env=RUN_ENV)
manifest = "requirements-modal.txt" if INSTALL_OPTIONAL_FEATURES else "requirements-core.txt"
run_logged(pip_install + ["-r", str(WORKSPACE_DIR / manifest), "flashinfer-python", "json-repair", "yt-dlp[default]"],
           cwd=WORKSPACE_DIR, env=RUN_ENV)
if INSTALL_OPTIONAL_FEATURES:
    # Pin the audited Stable Audio source without replacing the constrained Torch stack.
    run_logged([str(PYTHON), "-m", "pip", "install", "--force-reinstall", "--no-deps", "--ignore-requires-python",
                "git+https://github.com/Stability-AI/stable-audio-tools.git@3241adba4fc2a85cf5b29d9eb68d42f40a28e820"], env=RUN_ENV)
    # Qwen-TTS and audio-separator share ONNX import paths: GPU bindings go last.
    run_logged(pip_install + ["--reinstall", "--no-deps", "onnxruntime-gpu"], env=RUN_ENV)
if INSTALL_FLASH_ATTENTION:
    abi = subprocess.check_output(
        [str(PYTHON), "-c", "import torch; print(str(torch._C._GLIBCXX_USE_CXX11_ABI).upper())"],
        text=True, env=RUN_ENV,
    ).strip()
    wheel = (
        "https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.3.post1/"
        f"flash_attn-2.8.3.post1+cu12torch2.8cxx11abi{abi}-cp312-cp312-linux_x86_64.whl"
    )
    run_logged(pip_install + ["--no-deps", wheel], env=RUN_ENV)
def check_dependencies(python, env):
    """Keep pip check strict except for six audited Stable Audio metadata overrides."""
    result = subprocess.run(
        [str(python), "-m", "pip", "check"], env=env,
        capture_output=True, text=True,
    )
    if result.returncode == 0:
        print(result.stdout.strip(), flush=True)
        return
    # Stable Audio is installed at the pinned source revision with --no-deps.
    # These exact versions come from requirements-modal.txt / constraints-main.txt.
    overrides = {
        "importlib-resources": ("5.12.0", "7.1.0"),
        "PyWavelets": ("1.4.1", "1.10.0"),
        "sentencepiece": ("0.1.99", "0.2.2"),
        "torch": ("2.7.1", "2.8.0+cu128"),
        "torchaudio": ("2.7.1", "2.8.0+cu128"),
        "vector-quantize-pytorch": ("1.14.41", "1.31.6"),
    }
    allowed = {
        (f"stable-audio-tools 0.0.20 has requirement {name}=={required}, "
         f"but you have {name} {installed}.").casefold()
        for name, (required, installed) in overrides.items()
    }
    lines = [line.strip() for line in result.stdout.splitlines() if line.strip()]
    unexpected = [line for line in lines if line.casefold() not in allowed]
    if result.returncode != 1 or not lines or unexpected or result.stderr.strip():
        print(result.stdout, end="", flush=True)
        print(result.stderr, end="", flush=True)
        raise subprocess.CalledProcessError(result.returncode, result.args)
    for line in lines:
        print(f"[Known Stable Audio metadata override] {line}", flush=True)
    print("Dependency check passed with the six documented Stable Audio overrides.", flush=True)

check_dependencies(PYTHON, RUN_ENV)
run_logged([str(PYTHON), "-c", """
import importlib.metadata as metadata
import torch
assert metadata.version("vllm") == "0.10.2"
assert torch.__version__.split("+")[0] == "2.8.0"
assert torch.version.cuda == "12.8"
assert torch.cuda.is_available(), "CUDA is unavailable in the isolated environment"
x = torch.randn(64, 64, device="cuda")
print("CUDA smoke test:", (x @ x).mean().item())
print("Torch:", torch.__version__, "CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0), "capability:", torch.cuda.get_device_capability(0))
from indextts.infer_vllm_v2 import IndexTTS2
print("IndexTTS/vLLM imports OK")
"""], cwd=WORKSPACE_DIR, env=RUN_ENV)
print(f"Python environment ready: {PYTHON}")

def setup_clearvoice(enabled):
    # ClearVoice's rotary-embedding-torch pin conflicts with main audio separation.
    # Use its own manifest and interpreter, without main constraints/site-packages.
    RUN_ENV.pop("CLEARVOICE_PYTHON", None)
    if not enabled:
        print("ClearVoice setup disabled. Turn off reference enhancement and super-resolution in the UI.")
        return
    clearvoice_venv = NOTEBOOK_DIR / "venv_index_tts_clearvoice"
    clearvoice_python = clearvoice_venv / "bin/python"
    worker_env = RUN_ENV.copy()
    worker_env["VIRTUAL_ENV"] = str(clearvoice_venv)
    worker_env["PATH"] = str(clearvoice_venv / "bin") + os.pathsep + RUN_ENV["PATH"]
    if not clearvoice_python.is_file():
        run_logged(UV + ["venv", "--python", "3.12", "--seed", str(clearvoice_venv)], env=worker_env)
    version = subprocess.check_output(
        [str(clearvoice_python), "-c", "import sys; print('%s.%s' % sys.version_info[:2])"],
        text=True, env=worker_env,
    ).strip()
    if version != "3.12":
        raise RuntimeError(f"{clearvoice_venv} uses Python {version}; choose a fresh ClearVoice environment.")
    run_logged(UV + [
        "pip", "install", "--python", str(clearvoice_python), "--torch-backend", "cu128",
        "-r", str(WORKSPACE_DIR / "requirements-clearvoice.txt"),
    ], cwd=WORKSPACE_DIR, env=worker_env)
    run_logged([str(clearvoice_python), "-m", "pip", "check"], env=worker_env)
    run_logged([str(clearvoice_python), "-c", """
from clearvoice import ClearVoice
import importlib.metadata as metadata
import torch
assert metadata.version("rotary-embedding-torch") == "0.8.3"
assert torch.__version__.split("+")[0] == "2.8.0"
assert torch.version.cuda == "12.8"
assert torch.cuda.is_available(), "CUDA is unavailable in the ClearVoice environment"
print("ClearVoice imports OK; CUDA:", torch.version.cuda)
"""], cwd=WORKSPACE_DIR, env=worker_env)
    run_logged([str(clearvoice_python), "-m", "indextts_web.services.audio.clearvoice_worker", "--help"],
               cwd=WORKSPACE_DIR, env=worker_env)
    RUN_ENV["CLEARVOICE_PYTHON"] = str(clearvoice_python)
    print(f"ClearVoice worker ready: {clearvoice_python}")

setup_clearvoice(INSTALL_CLEARVOICE)


def setup_asr(kind, enabled):
    # These environments must not inherit TTS packages or its Transformers pin.
    keys = {
        "moss": ("COLAB_MOSS_PYTHON",),
        "qwen-asr": ("QWEN_OMNIVAD_PYTHON", "QWEN_OMNIVAD_MODEL_DIR", "QWEN_OMNIVAD_CACHE_DIR"),
    }[kind]
    for key in keys:
        RUN_ENV.pop(key, None)
    if kind == "moss":
        # Colab has no Docker daemon; only use the local service launched in cell 6.
        RUN_ENV.update({"MOSS_TRANSCRIBE_BACKEND": "http", "MOSS_TRANSCRIBE_MANAGE_BACKEND": "0",
                        "MOSS_TRANSCRIBE_SGLANG_URL": "http://127.0.0.1:8003",
                        "MOSS_TRANSCRIBE_DEVICE": "cuda:0",
                        "MOSS_TRANSCRIBE_MODEL": "OpenMOSS-Team/MOSS-Transcribe-Diarize"})
    if not enabled:
        print(f"{kind} setup disabled. Select another transcription backend in the UI.")
        return
    venv = NOTEBOOK_DIR / ("venv_index_tts_" + kind.replace("-", "_"))
    python = venv / "bin/python"
    env = RUN_ENV.copy()
    env["VIRTUAL_ENV"] = str(venv)
    env["PATH"] = str(venv / "bin") + os.pathsep + RUN_ENV["PATH"]
    if not python.is_file():
        run_logged(UV + ["venv", "--python", "3.12", "--seed", str(venv)], env=env)
    version = subprocess.check_output(
        [str(python), "-c", "import sys; print('%s.%s' % sys.version_info[:2])"], text=True, env=env,
    ).strip()
    if version != "3.12":
        raise RuntimeError(f"{venv} uses Python {version}; choose a fresh {kind} environment.")
    install = UV + ["pip", "install", "--python", str(python), "--torch-backend", "cu128"]
    if kind == "qwen-asr":
        # Reuse the vetted ASR/NeMo pins, replacing only the TTS Transformers pin.
        CACHE_DIR.mkdir(parents=True, exist_ok=True)
        asr_constraints = CACHE_DIR / "constraints-qwen-asr.txt"
        asr_constraints.write_text("".join(
            line for line in (WORKSPACE_DIR / "constraints-main.txt").read_text().splitlines(keepends=True)
            if not line.startswith("transformers==")
        ), encoding="utf-8")
        install += ["-c", str(asr_constraints)]
    run_logged(install + ["-r", str(WORKSPACE_DIR / f"requirements-colab-{kind}.txt")],
               cwd=WORKSPACE_DIR, env=env)
    run_logged([str(python), "-m", "pip", "check"], env=env)
    smoke = """
import torch
assert torch.__version__.split("+")[0] == "2.8.0"
assert torch.version.cuda == "12.8"
assert torch.cuda.is_available(), "CUDA is unavailable in the ASR environment"
"""
    if kind == "moss":
        smoke += """
from moss_transcribe_diarize.inference_utils import generate_transcription
from moss_transcribe_server import app, runtime
assert runtime.status()["state"] == "unloaded"
print("MOSS service imports OK (weights load on first use)")
"""
    else:
        smoke += """
import importlib.metadata as metadata
from qwen_asr import Qwen3ASRModel
from omnivad import OmniVAD
from nemo.collections.asr.models import SortformerEncLabelModel
import qwen_omnivad_pipeline as pipeline
assert metadata.version("transformers") == "4.57.6"
assert pipeline.is_qwen_omnivad_available()
assert pipeline._make_translation_llm is not None
assert pipeline.DiarizationPipeline is not None
print("Qwen3-ASR, OmniVAD, diarization and translation imports OK")
"""
    run_logged([str(python), "-c", smoke], cwd=WORKSPACE_DIR, env=env)
    if kind == "qwen-asr":
        run_logged([str(python), "-m", "indextts_web.services.translation.qwen_worker", "--help"],
                   cwd=WORKSPACE_DIR, env=env)
        RUN_ENV.update({"QWEN_OMNIVAD_PYTHON": str(python),
                        "QWEN_OMNIVAD_MODEL_DIR": str(WORKSPACE_DIR / "checkpoints/qwen_omnivad"),
                        "QWEN_OMNIVAD_CACHE_DIR": str(CACHE_DIR / "qwen_omnivad")})
    else:
        RUN_ENV["COLAB_MOSS_PYTHON"] = str(python)
    print(f"{kind} environment ready: {python}")

setup_asr("moss", INSTALL_MOSS)
setup_asr("qwen-asr", INSTALL_QWEN_ASR)


### 4. Optional Google Drive persistence

Enable to save outputs and speaker presets under MyDrive/IndexTTS-vLLM.
Checkpoints stay on runtime disk. Before linking a populated local folder, the notebook copies missing data to Drive and preserves the original folder as a local backup.
If both locations contain different copies of the same file, setup stops instead of overwriting either copy.
Run this cell before launching the app.

In [ ]:
# @title 4. (Optional) Save outputs and speaker presets to Google Drive
MOUNT_GOOGLE_DRIVE = False  # @param {type:"boolean"}
DRIVE_FOLDER = "IndexTTS-vLLM"  # @param {type:"string"}

def merge_missing_files(source, destination):
    import filecmp
    for item in source.iterdir():
        target = destination / item.name
        if item.is_symlink():
            raise RuntimeError(f"Review local symlink before migration: {item}")
        if item.is_dir():
            target.mkdir(parents=True, exist_ok=True)
            merge_missing_files(item, target)
        elif target.exists():
            if not target.is_file() or not filecmp.cmp(item, target, shallow=False):
                raise RuntimeError(f"Drive conflict: {target}. Resolve it before rerunning; local files are preserved.")
        else:
            shutil.copy2(item, target)

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    mount_dir = NOTEBOOK_DIR / "drive"
    drive.mount(str(mount_dir))
    my_drive = (mount_dir / "MyDrive").resolve()
    DRIVE_ROOT = (my_drive / DRIVE_FOLDER).resolve()
    if DRIVE_ROOT == my_drive or not DRIVE_ROOT.is_relative_to(my_drive):
        raise ValueError("DRIVE_FOLDER must name a subfolder inside MyDrive.")
    for name in ("outputs", "speaker_presets"):
        local = WORKSPACE_DIR / name
        target = DRIVE_ROOT / name
        if local.is_symlink():
            if local.resolve() != target.resolve():
                raise RuntimeError(f"{local} already links elsewhere; restore or choose the original Drive folder.")
            print(f"Reusing {local} -> {target}")
            continue
        if local.exists() and not local.is_dir():
            raise RuntimeError(f"Expected a directory: {local}")
        target.mkdir(parents=True, exist_ok=True)
        if local.exists():
            merge_missing_files(local, target)
            backup = local.with_name(f"{name}-before-drive-{uuid.uuid4().hex[:8]}")
            local.rename(backup)
            print(f"Preserved local backup: {backup}")
        local.symlink_to(target, target_is_directory=True)
        print(f"{local} -> {target}")
else:
    for name in ("outputs", "speaker_presets"):
        local = WORKSPACE_DIR / name
        if local.is_symlink():
            raise RuntimeError(f"{local} is linked to Drive. Re-enable Drive mounting before continuing.")
        local.mkdir(parents=True, exist_ok=True)
    print("Using local runtime storage.")


### 5. Download model weights

Downloads are resumable through Hugging Face's local-directory metadata.
The main IndexTTS 2.0 bundle is required. HY-MT translation is optional.
For authenticated downloads, add a Colab secret named HF_TOKEN, allow notebook access, and enable USE_HF_SECRET.
Tokens are passed through the child environment and never written to notebook source or launcher files.
Some auxiliary weights may still download on first inference.

In [ ]:
# @title 5. Download IndexTTS 2.0 and optional translation weights
DOWNLOAD_HY_MT = True  # @param {type:"boolean"}
USE_HF_SECRET = False  # @param {type:"boolean"}

if USE_HF_SECRET:
    from google.colab import userdata
    RUN_ENV["HF_TOKEN"] = userdata.get("HF_TOKEN")

MODEL_DIR = WORKSPACE_DIR / "checkpoints"
run_logged([str(PYTHON), "-c", """
from huggingface_hub import snapshot_download
import sys
snapshot_download("garyswansrs/index_tts_2_vllm", local_dir=sys.argv[1])
""", str(MODEL_DIR)], cwd=WORKSPACE_DIR, env=RUN_ENV)
if DOWNLOAD_HY_MT:
    run_logged([str(PYTHON), "-c", """
from huggingface_hub import snapshot_download
import sys
snapshot_download("tencent/Hy-MT2-1.8B", local_dir=sys.argv[1])
""", str(MODEL_DIR / "hy-mt")], cwd=WORKSPACE_DIR, env=RUN_ENV)
RUN_ENV["HY_MT_TRANSLATION_LOCAL_DIR"] = str(MODEL_DIR / "hy-mt")
run_logged([str(PYTHON), "-c", """
from pathlib import Path
import sys
root = Path(sys.argv[1])
for name in ("config.yaml", "s2mel.pth", "gpt/config.json"):
    assert (root / name).is_file(), f"Missing model file: {root / name}"
assert list((root / "gpt").glob("model*.safetensors")) or list((root / "gpt").glob("pytorch_model*.bin")), "Missing GPT weights"
print("Required IndexTTS checkpoint files are present.")
""", str(MODEL_DIR)], env=RUN_ENV)


### 6. Launch the speech studio

Cloudflare is enabled by default. The launch cell waits for /health to report ready, then prints a trycloudflare.com URL.
[Cloudflare Quick Tunnels](https://developers.cloudflare.com/tunnel/get-started/quick-tunnels/) are temporary; restarting produces a new URL.
If you disable the public tunnel, use Colab's authenticated port preview.

The app chooses its memory budget from detected VRAM.
Warmup is optional and **off by default**. Enable ENABLE_WARMUP to speed up generation,
but it costs time to warm up before the app becomes ready. This single option controls S2Mel compilation and startup warmup together: both off or both on.
Interrupt this cell to stop the app, workers, and tunnel before rerunning setup.

For Colab Terminal use, this cell also writes **run_index_tts_colab.sh** in the workspace.
Stop the notebook launch first, then run that script from the printed workspace path.
The terminal launcher reuses the environment and files; it does not contain the HF token.

In [ ]:
# @title 6. Launch IndexTTS-vLLM with a public URL
ENABLE_CLOUDFLARE_TUNNEL = True  # @param {type:"boolean"}
# @markdown One switch for warmup and S2Mel compilation: both off by default. Turning it on speeds up generation but costs startup warmup time.
ENABLE_WARMUP = False  # @param {type:"boolean"}
SERVER_PORT = 8000  # @param {type:"integer"}
STARTUP_TIMEOUT_SECONDS = 1800  # @param {type:"integer"}

import shlex

if not 1 <= SERVER_PORT <= 65535:
    raise ValueError("SERVER_PORT must be between 1 and 65535.")
if STARTUP_TIMEOUT_SECONDS < 1:
    raise ValueError("STARTUP_TIMEOUT_SECONDS must be positive.")
for name in ("outputs", "speaker_presets"):
    local = WORKSPACE_DIR / name
    if not local.is_dir():
        raise RuntimeError(f"{local} is unavailable. Rerun cell 4, including Drive mounting if enabled.")

# Keep the app, automatic GPU profile, and child processes on the same setting.
RUN_ENV["INDEXTTS_USE_TORCH_COMPILE"] = str(int(ENABLE_WARMUP))
COLAB_DIR = WORKSPACE_DIR / ".colab"
COLAB_DIR.mkdir(exist_ok=True)
CLOUDFLARED = COLAB_DIR / "cloudflared"
if ENABLE_CLOUDFLARE_TUNNEL and not CLOUDFLARED.is_file():
    partial = CLOUDFLARED.with_suffix(".partial")
    run_logged(["curl", "--fail", "--location", "--retry", "3",
                "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                "--output", str(partial)], env=RUN_ENV)
    partial.chmod(0o755)
    partial.replace(CLOUDFLARED)

LAUNCHER_SOURCE = r'''
import json
import os
from pathlib import Path
import queue
import signal
import socket
import subprocess
import sys
import threading
import time
import urllib.error
import urllib.request

workspace = Path(__file__).resolve().parent.parent
port = int(sys.argv[1])
public = sys.argv[2] == "1"
warmup_enabled = sys.argv[3] == "1"
startup_timeout = int(sys.argv[4])
events = queue.Queue()
processes = []

def stop(proc):
    try:
        os.killpg(proc.pid, signal.SIGTERM)
    except ProcessLookupError:
        return
    try:
        proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        pass
    try:
        os.killpg(proc.pid, signal.SIGKILL)
    except ProcessLookupError:
        pass
    proc.wait(timeout=5)

def relay(proc, label):
    for line in proc.stdout:
        events.put((label, line))
    proc.stdout.close()

def start(command, label):
    proc = subprocess.Popen(
        command, cwd=workspace, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace", bufsize=1, start_new_session=True,
    )
    processes.append(proc)
    threading.Thread(target=relay, args=(proc, label), daemon=True).start()
    return proc

def server_command():
    # Override any inherited compilation setting with the notebook's one switch.
    os.environ["INDEXTTS_USE_TORCH_COMPILE"] = str(int(warmup_enabled))
    return [
        sys.executable, "-u", str(workspace / "fastapi_webui_v2.py"),
        "--host", "0.0.0.0", "--port", str(port),
        "--model_dir", str(workspace / "checkpoints"), "--tts_backend", "index",
        "--use_torch_compile" if warmup_enabled else "--no-use_torch_compile",
        "--no-confucius_detach_process", "--no-indextts25_detach_process",
    ]

def start_moss_service():
    python = os.environ.get("COLAB_MOSS_PYTHON")
    if not python:
        return None
    moss_port = 8003
    if port == moss_port:
        raise ValueError("SERVER_PORT 8003 is reserved for the local MOSS service. Choose another WebUI port.")
    with socket.socket() as probe:
        probe.bind(("127.0.0.1", moss_port))
    # The same process list owns MOSS, the app and the tunnel, including failed startup.
    proc = start([python, "-m", "uvicorn", "moss_transcribe_server:app",
                  "--host", "127.0.0.1", "--port", str(moss_port)], "moss")
    deadline = time.monotonic() + min(startup_timeout, 120)
    while time.monotonic() < deadline:
        try:
            label, line = events.get(timeout=0.2)
            print(f"[{label}] {line}", end="", flush=True)
        except queue.Empty:
            pass
        if proc.poll() is not None:
            raise RuntimeError(f"MOSS service exited with code {proc.returncode}; inspect the moss logs above.")
        try:
            with urllib.request.urlopen("http://127.0.0.1:8003/model/status", timeout=1) as response:
                status = json.load(response)
                if status.get("service") == "moss-transcribe":
                    print("MOSS service ready; weights load on first transcription or model wake.", flush=True)
                    return proc
        except (OSError, ValueError, urllib.error.HTTPError):
            pass
    raise TimeoutError("MOSS service startup timed out; inspect the moss logs above.")


def interrupted(signum, frame):
    raise KeyboardInterrupt

# Catch outer notebook cancellation so separately grouped app workers are cleaned up.
signal.signal(signal.SIGTERM, interrupted)
signal.signal(signal.SIGINT, interrupted)
try:
    with socket.socket() as probe:
        probe.bind(("127.0.0.1", port))
    moss_server = start_moss_service()
    command = server_command()
    state = "ON" if warmup_enabled else "OFF"
    print(f"Warmup and S2Mel compilation: {state}", flush=True)
    server = start(command, "app")
    tunnel = None
    ready = False
    started = time.monotonic()
    next_check = 0
    print("Starting IndexTTS. Keep this cell running; stop it to close the app.", flush=True)
    while True:
        try:
            label, line = events.get(timeout=0.2)
            print(f"[{label}] {line}", end="", flush=True)
        except queue.Empty:
            pass
        if moss_server is not None and moss_server.poll() is not None:
            raise RuntimeError(f"MOSS service exited with code {moss_server.returncode}; inspect the moss logs above.")
        if server.poll() is not None:
            raise RuntimeError(f"WebUI exited with code {server.returncode}; inspect the app logs above.")
        if tunnel is not None and tunnel.poll() is not None:
            raise RuntimeError(f"Cloudflare exited with code {tunnel.returncode}; inspect the tunnel logs above.")
        now = time.monotonic()
        if not ready and now >= next_check:
            next_check = now + 3
            try:
                with urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=1) as response:
                    ready = bool(json.load(response).get("ready"))
            except (OSError, ValueError, urllib.error.HTTPError):
                pass
            if ready:
                print(f"\nWebUI ready at http://127.0.0.1:{port}", flush=True)
                if public:
                    tunnel = start([
                        str(workspace / ".colab/cloudflared"), "tunnel",
                        "--url", f"http://127.0.0.1:{port}", "--no-autoupdate",
                        "--protocol", "http2",
                    ], "tunnel")
                    print("Open the https://...trycloudflare.com URL in the tunnel logs below.", flush=True)
            elif now - started > startup_timeout:
                raise TimeoutError("Model startup timed out. Inspect the app logs or increase the timeout.")
except KeyboardInterrupt:
    print("\nStopping MOSS, IndexTTS and the public tunnel...", flush=True)
finally:
    # Do not let a second notebook stop interrupt cleanup.
    signal.signal(signal.SIGTERM, signal.SIG_IGN)
    signal.signal(signal.SIGINT, signal.SIG_IGN)
    for proc in reversed(processes):
        stop(proc)
'''
(COLAB_DIR / "serve.py").write_text(LAUNCHER_SOURCE, encoding="utf-8")
launcher = WORKSPACE_DIR / "run_index_tts_colab.sh"
persistent_env = {
    key: RUN_ENV[key] for key in (
        "HF_HOME", "TORCH_HOME", "TRITON_CACHE_DIR", "VLLM_CACHE_ROOT",
        "HY_MT_TRANSLATION_LOCAL_DIR", "PYTHONUNBUFFERED", "PYTHONNOUSERSITE",
        "CLEARVOICE_PYTHON", "INDEXTTS_USE_TORCH_COMPILE",
        "COLAB_MOSS_PYTHON", "MOSS_TRANSCRIBE_BACKEND", "MOSS_TRANSCRIBE_MANAGE_BACKEND",
        "MOSS_TRANSCRIBE_SGLANG_URL", "MOSS_TRANSCRIBE_DEVICE", "MOSS_TRANSCRIBE_MODEL",
        "QWEN_OMNIVAD_PYTHON", "QWEN_OMNIVAD_MODEL_DIR", "QWEN_OMNIVAD_CACHE_DIR",
    ) if key in RUN_ENV
}
launcher_text = "#!/usr/bin/env bash\nset -euo pipefail\n"
launcher_text += "unset PYTHONPATH PYTHONHOME MPLBACKEND CLEARVOICE_PYTHON COLAB_MOSS_PYTHON QWEN_OMNIVAD_PYTHON\n"
launcher_text += f"export VIRTUAL_ENV={shlex.quote(str(VENV_DIR))}\n"
launcher_text += f'export PATH={shlex.quote(str(VENV_DIR / "bin"))}:"$PATH"\n'
for key, value in persistent_env.items():
    launcher_text += f"export {key}={shlex.quote(value)}\n"
launcher_text += "exec " + shlex.join([
    str(PYTHON), "-u", str(COLAB_DIR / "serve.py"), str(SERVER_PORT),
    str(int(ENABLE_CLOUDFLARE_TUNNEL)), str(int(ENABLE_WARMUP)),
    str(STARTUP_TIMEOUT_SECONDS),
]) + "\n"
launcher.write_text(launcher_text, encoding="utf-8")
launcher.chmod(0o755)
print(f"Terminal launcher: bash {shlex.quote(str(launcher))}", flush=True)
if not ENABLE_CLOUDFLARE_TUNNEL:
    from google.colab import output
    output.serve_kernel_port_as_window(SERVER_PORT)
run_logged(["bash", str(launcher)], cwd=WORKSPACE_DIR, env=RUN_ENV)


### Using the studio

- Open the public URL, upload a short reference voice or save a speaker preset, then enter text and synthesize with **IndexTTS 2.0**.
- Download generated files from the UI. When Drive is enabled, outputs and presets persist under MyDrive/IndexTTS-vLLM.
- Optional main-process integrations include WhisperX/Parakeet transcription, audio separation, video downloads, Qwen3 voice design, and Stable Audio runtime dependencies. Their weights may download on first use; gated Stable Audio weights need separate authorization and downloading.
- ClearVoice reference enhancement and super-resolution use the isolated worker installed by default in cell 3. First use downloads its weights.
- MOSS and Qwen3-ASR are enabled by default in cell 3. Cell 6 starts the local Docker-free MOSS service and configures the Qwen worker. First use downloads their weights and can take several minutes.
- Confucius and IndexTTS 2.5 are not provisioned; see README_EN.md for their setup.
- After reconnecting to a new runtime, rerun cells 1–6. To restart within the same runtime, stop cell 6 and rerun it. To change Drive settings, stop the app first.
- If setup fails, fix the first reported error and rerun that cell. The notebook does not modify quickstart.sh or upgrade vLLM beyond the repository pin.